# Spike — validação de sys.path dinâmico em mode: production

Testa se dbutils.notebook.entry_point (mecanismo baseado em JVM) funciona no compute serverless (Spark Connect) — nunca testado neste ambiente. Se falhar, usamos caminho fixo como alternativa.

Motivado por: job_diario falhou com ModuleNotFoundError após migração para mode: production (source-linked deployment desativado).

In [0]:
%pip install dbldatagen Faker

In [0]:
dbutils.library.restartPython()

In [0]:
try:
    notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    print("Funcionou! Caminho do notebook:", notebook_path)
except Exception as e:
    print("Falhou:", type(e).__name__, "-", e)

In [0]:
import sys

notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()

if "/files/" in notebook_path:
    # rodando via Job (Bundle implantado, mode: production)
    root_path = notebook_path.split("/files/")[0] + "/files"
else:
    # rodando direto no código-fonte (edição manual, mode: development)
    root_path = notebook_path.rsplit("/src/", 1)[0]

if root_path not in sys.path:
    sys.path.append(root_path)

print("Root adicionado ao sys.path:", root_path)

In [0]:
from src.simuladores.simulador_factory import SimuladorFactory

print("Import funcionou! Ordem de execução:", SimuladorFactory.ordem_execucao())